<a href="https://colab.research.google.com/github/dataprogpy/code-samples/blob/main/notebooks/altair.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Statements

In [1]:
import polars as pl
import altair as alt
from sklearn.cluster import MeanShift, estimate_bandwidth

# Mounting Google Drive

In [4]:
from google.colab import drive
drive.mount('/gdrive')

Drive already mounted at /gdrive; to attempt to forcibly remount, call drive.mount("/gdrive", force_remount=True).
/gdrive/My Drive/dataprogpy
ls: cannot access '/data': No such file or directory


# Listing Files From Google Drive Folder

In [10]:
%cd "/gdrive/My Drive/dataprogpy/data"
!ls .

/gdrive/My Drive/dataprogpy/data
cars.json


In [11]:
df = pl.read_json("/gdrive/My Drive/dataprogpy/data/cars.json")
df.head()

Name,Miles_per_Gallon,Cylinders,Displacement,Horsepower,Weight_in_lbs,Acceleration,Year,Origin
str,i64,i64,f64,i64,i64,f64,str,str
"""chevrolet chevelle malibu""",18,8,307.0,130,3504,12.0,"""1970-01-01""","""USA"""
"""buick skylark 320""",15,8,350.0,165,3693,11.5,"""1970-01-01""","""USA"""
"""plymouth satellite""",18,8,318.0,150,3436,11.0,"""1970-01-01""","""USA"""
"""amc rebel sst""",16,8,304.0,150,3433,12.0,"""1970-01-01""","""USA"""
"""ford torino""",17,8,302.0,140,3449,10.5,"""1970-01-01""","""USA"""


In [16]:
df.select(pl.col("Origin").unique())

Origin
str
"""USA"""
"""Japan"""
"""Europe"""


In [54]:
Countries = pl.Enum(["Europe", "Japan", "USA"])
cars = df.with_columns(
    pl.col("Origin").cast(Countries),
    pl.col("Year").cast(pl.Date)
    )
cars.head()

Name,Miles_per_Gallon,Cylinders,Displacement,Horsepower,Weight_in_lbs,Acceleration,Year,Origin
str,i64,i64,f64,i64,i64,f64,date,enum
"""chevrolet chevelle malibu""",18,8,307.0,130,3504,12.0,1970-01-01,"""USA"""
"""buick skylark 320""",15,8,350.0,165,3693,11.5,1970-01-01,"""USA"""
"""plymouth satellite""",18,8,318.0,150,3436,11.0,1970-01-01,"""USA"""
"""amc rebel sst""",16,8,304.0,150,3433,12.0,1970-01-01,"""USA"""
"""ford torino""",17,8,302.0,140,3449,10.5,1970-01-01,"""USA"""


#  Visualization: Interactive Scatter Plot in Altair

Altair lets you easily create an interactive scatter plot from data stored in a Pandas dataframe.

In [35]:
# load an example dataset
# from vega_datasets import data
# cars = data.cars()

# plot the dataset, referencing dataframe column names
# import altair as alt
alt.Chart(cars).mark_point().encode(
  x='Horsepower',
  y='Miles_per_Gallon',
  color='Origin'
).interactive()

alt.Chart(...)

# Visualization: Bar Plot in Altair

This shows a simple bar plot in Altair, showing the mean miles per gallon as a function of origin for a number of car models:

In [23]:
alt.Chart(cars).mark_bar().encode(
  x='mean(Miles_per_Gallon)',
  y='Origin',
  color='Origin'
)

alt.Chart(...)

In [33]:
alt.Chart(cars).mark_line().encode(
    x="Year",
    y="mean(Miles_per_Gallon)",
    color="Origin:N"
)

alt.Chart(...)

In [24]:
alt.Chart(cars).mark_bar().encode(
  x='mean(Miles_per_Gallon)',
  y='Origin',
  color='Origin'
)

alt.Chart(...)

# Visualization: Histogram in Altair

Altair provides a variety of aggregation operations in order to build custom histograms. Here is a simple example


In [25]:
alt.Chart(cars).mark_bar().encode(
  x=alt.X('Miles_per_Gallon', bin=True),
  y='count()',
)

alt.Chart(...)

# Visualization: Stacked Histogram in Altair

If you take a standard histogram and encode another field with color, the result will be a stacked histogram:


In [32]:
alt.Chart(cars).mark_bar().encode(
  x=alt.X('Miles_per_Gallon', bin=True),
  y='count()',
  color='Origin:N'
)

alt.Chart(...)

# Visualization: Scatter Plot with Rolling Mean in Altair

This shows a scatter chart of miles per gallon as a function of year, with lines inidicating the mean values for each country within the given year.

In [27]:
points = alt.Chart(cars).mark_point().encode(
  x='Year:T',
  y='Miles_per_Gallon',
  color='Origin:N'
).properties(
  width=800
)

lines = alt.Chart(cars).mark_line().encode(
  x='Year:T',
  y='mean(Miles_per_Gallon)',
  color='Origin'
).properties(
  width=800
).interactive(bind_y=False)

points + lines

alt.LayerChart(...)

In [53]:
cars.head()

,Name,Miles_per_Gallon,Cylinders,Displacement,Horsepower,Weight_in_lbs,Acceleration,Year,Origin
0,chevrolet chevelle malibu,18.0,8,307.0,130.0,3504,12.0,1970-01-01,USA
1,buick skylark 320,15.0,8,350.0,165.0,3693,11.5,1970-01-01,USA
2,plymouth satellite,18.0,8,318.0,150.0,3436,11.0,1970-01-01,USA
3,amc rebel sst,16.0,8,304.0,150.0,3433,12.0,1970-01-01,USA
4,ford torino,17.0,8,302.0,140.0,3449,10.5,1970-01-01,USA


In [56]:
from datetime import date
date1 = date(1975, 1, 1)
date2 = date(1980, 12, 31)
yrs_75_80 = cars.filter(
        pl.col("Year").is_between(date1, date2)
        )
points = alt.Chart(
    yrs_75_80
    ).mark_point().encode(
  x='Year:T',
  y='Miles_per_Gallon',
  color='Origin:N'
).properties(
  width=800
)

lines = alt.Chart(
    yrs_75_80
    ).mark_line().encode(
  x='Year:T',
  y='mean(Miles_per_Gallon)',
  color='Origin'
).properties(
  width=800
).interactive(bind_y=False)

points + lines

alt.LayerChart(...)

#  Visualization: Interactive Brushing in Altair

With a few extra lines of code on top of a standard scatter plot, you can add selection behavior to your scatter plot. This lets you click and drag to select points.

In [28]:
interval = alt.selection_interval()

alt.Chart(cars).mark_point().encode(
  x='Horsepower',
  y='Miles_per_Gallon',
  color=alt.condition(interval, 'Origin', alt.value('lightgray'))
).properties(
  selection=interval
)

SchemaValidationError: `Chart` has no parameter named 'selection'

Existing parameter names are:
data       mark    height   
encoding   width   kwargs   

See the help for `Chart` to read the full description of these parameters

alt.Chart(...)

In [36]:
interval = alt.selection_interval()

alt.Chart(cars).mark_point().encode(
  x='Horsepower',
  y='Miles_per_Gallon',
  color=alt.condition(interval, 'Origin', alt.value('lightgray'))
).add_selection(
  interval
)

<ipython-input-36-6c8c7cc93cee>:7: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  ).add_selection(


alt.Chart(...)

# Visualization: Linked Brushing in Altair

If you apply the same selection to multiple panels of an Altair chart, the selections will be linked:

In [39]:
# load an example dataset
from vega_datasets import data
cars = data.cars()

import altair as alt

interval = alt.selection_interval()

base = alt.Chart(cars).mark_point().encode(
  y='Miles_per_Gallon',
  color=alt.condition(interval, 'Origin', alt.value('lightgray'))
).add_selection(
  interval
)

base.encode(x='Acceleration') | base.encode(x='Horsepower')

<ipython-input-39-de79fe75936a>:12: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  ).add_selection(


alt.HConcatChart(...)

# Visualization: Linked Scatter-Plot and Histogram in Altair

Altair selections can be used for a variety of things. This example shows a scatter plot and a histogram with selections over both that allow exploring the relationships between points

In [37]:
interval = alt.selection_interval()

points = alt.Chart(cars).mark_point().encode(
  x='Horsepower',
  y='Miles_per_Gallon',
  color=alt.condition(interval, 'Origin', alt.value('lightgray'))
).add_selection(
  interval
)

histogram = alt.Chart(cars).mark_bar().encode(
  x='count()',
  y='Origin',
  color='Origin'
).transform_filter(interval)

points & histogram

<ipython-input-37-427e439be369>:7: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use add_params instead.
  ).add_selection(


alt.VConcatChart(...)

# Visualization: Time Series Line Plot in Altair

Altair handles temporal types natively by using the ``:T`` type marker. An example is in this plot of stock prices over time

In [38]:
from vega_datasets import data
stocks = data.stocks()

alt.Chart(stocks).mark_line().encode(
  x='date:T',
  y='price',
  color='symbol'
).interactive(bind_y=False)

alt.Chart(...)